The purpose of this file is to verify the correctness of the sorted insertion implementation. To do this, I will check if I get the same variance values for the Hamiltonians in `ham_lib` compared to `reference tbd`.

In [3]:
import sys
sys.path.append('../')

from utils_m4_partitioning import (
    sorted_insertion_decomposition,
    abs_of_dict_value
)
from utils_results import variance_of_decomp
from utils_states import convert_dense_format_to_sparse_format
import pickle
import numpy as np
from openfermion import (
    bravyi_kitaev,
    get_sparse_operator,
    variance,
    get_ground_state
)

In [6]:
# recover the results from previous works

N_QUBITS = {
    'h2'   : 4,
    'lih'  : 12,
    'beh2' : 14,
    'h2o'  : 14,
    'nh3'  : 16
}


for moltag in N_QUBITS:
    Nqubits = N_QUBITS[moltag]

    filename = f'ham_lib/{moltag}_fer.bin'
    with open(filename, 'rb') as f:
        Hfer = pickle.load(f)
    Hqub  = bravyi_kitaev(Hfer)
    Hqub -= Hqub.constant
    Hqub.compress()
    Hqub.terms = dict(sorted(Hqub.terms.items(), key=abs_of_dict_value, reverse=True))

    psi_GS = convert_dense_format_to_sparse_format(get_ground_state(get_sparse_operator(Hqub, Nqubits))[1]) 
    
    methodtag  = 'fc'
    decomp     = sorted_insertion_decomposition(Hqub, methodtag)
    var_metric = variance_of_decomp(decomp, psi_GS, Nqubits, general=True)

    print(f'''
        Molecule : {moltag}
        Method   : {methodtag}
        Result   : {var_metric}

    ''')


        Molecule : h2
        Method   : fc
        Result   : (0.13644847184849931+4.3908053096660844e-17j)

    

        Molecule : lih
        Method   : fc
        Result   : (0.8816294035047314-1.3899484156451136e-15j)

    

        Molecule : beh2
        Method   : fc
        Result   : (1.1113315710462848-1.5306565558117285e-14j)

    

        Molecule : h2o
        Method   : fc
        Result   : (7.58895575182792-4.685473705483678e-13j)

    

        Molecule : nh3
        Method   : fc
        Result   : (18.75101999401598-4.104149241947144e-13j)

    


In [8]:
# my code should work for complex Hamiltonians --> multiply everything by 1j and see if the results stay the same
# Var_psi(1j * O) == Var_psi(O)

N_QUBITS = {
    'h2'   : 4,
    'lih'  : 12,
    'beh2' : 14,
    'h2o'  : 14,
    'nh3'  : 16
}


for moltag in N_QUBITS:
    Nqubits = N_QUBITS[moltag]

    filename = f'ham_lib/{moltag}_fer.bin'
    with open(filename, 'rb') as f:
        Hfer = pickle.load(f)
    Hqub  = bravyi_kitaev(Hfer)
    Hqub -= Hqub.constant
    Hqub.compress()
    Hqub.terms = dict(sorted(Hqub.terms.items(), key=abs_of_dict_value, reverse=True))

    psi_GS = convert_dense_format_to_sparse_format(get_ground_state(get_sparse_operator(Hqub, Nqubits))[1]) 
    
    methodtag  = 'fc'
    decomp     = sorted_insertion_decomposition(1j * Hqub, methodtag)
    var_metric = variance_of_decomp(decomp, psi_GS, Nqubits, general=True)

    print(f'''
        Molecule : {moltag}
        Method   : {methodtag}
        Result   : {var_metric}

    ''')


        Molecule : h2
        Method   : fc
        Result   : (0.13644847184849965+5.18200311239477e-17j)

    

        Molecule : lih
        Method   : fc
        Result   : (0.8816294035047801+1.7314700422123171e-15j)

    

        Molecule : beh2
        Method   : fc
        Result   : (1.1113315710447647+2.1568610674822255e-14j)

    

        Molecule : h2o
        Method   : fc
        Result   : (7.588955751750382+1.2992371815539362e-13j)

    

        Molecule : nh3
        Method   : fc
        Result   : (18.751019994006025+2.3623259843232313e-13j)

    
